# Revisión sistemática de literatura

In [54]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, HTML
from matplotlib.ticker import MultipleLocator

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as ts

## Introducción

Este cuaderno documenta la revisión sistemática que fundamenta la selección de los diez modelos comparados en el benchmark del proyecto, cuyo objetivo es el pronóstico de volatilidad del crudo Brent. 

El propósito de la revisión es identificar y justificar los modelos que serán comparados empíricamente. Para ello, la revisión describe las familias de arquitecturas aplicadas al problema (econométricas, de machine learning y de deep learning), valora en qué medida los estudios incluidos permiten una comparación justa y reproducible, y formula la brecha de comparabilidad que motiva el benchmark propio. La búsqueda sigue el protocolo PRISMA 2020 y se basa en Scopus, IEEE Xplore y Web of Science.

El cuaderno se organiza en cuatro secciones principales. La primera sección detalla el método de revisión, las preguntas de investigación, las cadenas de búsqueda, los criterios de elegibilidad y el proceso de cribado. La segunda sección presenta los resultados del corpus incluido, la categorización inductiva de las familias arquitectónicas y la matriz de extracción de datos. La tercera sección expone la evaluación del riesgo de sesgo de los 16 estudios. La cuarta sección define los 10 modelos del benchmark (abarcando enfoques econométricos, de machine learning y de deep learning) y formula la brecha de investigación que el proyecto aborda.

## Método de revisión

El protocolo de búsqueda y cribado empleado para construir el corpus del que se seleccionan los diez modelos comparados en el benchmark se describe en tres partes: las preguntas de investigación y las cadenas de búsqueda, los criterios de inclusión y exclusión, y el proceso de cribado.

### Preguntas de investigación y estrategia de búsqueda

El análisis aborda la pregunta general: ¿qué familias de arquitecturas de pronóstico se han aplicado al pronóstico de volatilidad del crudo Brent (o de activos energéticos y financieros directamente comparables) en la literatura científica con revisión por pares, y en qué medida permiten una comparación justa y reproducible? Esta pregunta se descompone en tres subpreguntas:

- **RQ1.** ¿Qué familias de arquitecturas (econométricas, de machine learning y de deep learning) se han aplicado al pronóstico de volatilidad del crudo (o de activos directamente comparables) en la literatura científica con revisión por pares?
- **RQ2.** ¿En qué medida los estudios incluidos informan de condiciones que permitan una comparación justa: partición temporal sin fuga, función de pérdida adecuada para volatilidad, pruebas de significancia entre modelos y disponibilidad de hiperparámetros o código?
- **RQ3.** ¿Qué lagunas de comparabilidad y reproducibilidad justifican la construcción de un benchmark propio con presupuesto de ajuste equivalente y pruebas estadísticas formales?

Para responder a estas preguntas, se emplearon dos consultas encadenadas por base de datos. La cadena principal exige proximidad entre cuatro bloques conceptuales: activo u objeto (A), tarea (B), método (C) y evaluación (D). Por su parte, la cadena puente elimina el bloque de tarea y restringe el bloque de método a aparecer cerca de un término explícito de series temporales financieras o volatilidad. Esta segunda cadena es fundamental porque permite capturar trabajos metodológicos que operan sobre pronóstico financiero sin autodenominarse explícitamente como "volatilidad", pero que son altamente transferibles al problema.

Los motores de búsqueda presentan limitaciones sintácticas que exigen adaptar las consultas a cada plataforma. Scopus y Web of Science utilizaron los operadores de proximidad estándar (W/n y NEAR/n, respectivamente). IEEE Xplore utilizó el operador lógico AND en la cadena principal (por falta de soporte para NEAR) y expandió la cadena puente en combinaciones explícitas por pares. Las ecuaciones de búsqueda exactas se encuentran disponibles para consulta en el repositorio del proyecto, en el archivo `review/summary.md`.

El periodo de búsqueda abarcó de 2018 a 2026, lo que cubre los últimos ocho años de producción científica sobre Deep Learning aplicado a volatilidad. La revisión admitió trabajos anteriores únicamente cuando introducen una arquitectura que sigue operando como modelo de referencia en estudios de 2020 a 2026. La matriz de extracción justifica esta excepción con al menos una cita de respaldo.

In [5]:
data = {
    "Aspecto": [
        "Cadena principal",
        "Cadena puente",
        "Operadores de proximidad",
        "Idiomas",
        "Rango temporal",
        "Bases de datos"
    ],
    "Especificación": [
        '("Brent" OR "crude oil" OR "oil price" OR "energy market") W/15 ("volatility" OR "realized volatility" OR "variance forecasting" OR "GARCH") W/15 ("deep learning" OR "neural network" OR "LSTM" OR "Transformer" OR "CNN" OR "hybrid model") W/15 ("forecasting" OR "prediction" OR "out-of-sample")',
        '("Brent" OR "crude oil" OR "oil price" OR "energy market") W/15 ("deep learning" OR "neural network" OR "LSTM" OR "Transformer" OR "CNN" OR "hybrid model") W/15 ("forecasting" OR "prediction" OR "out-of-sample") AND ("deep learning" OR "neural network") W/10 ("financial time series" OR "volatility")',
        "W/n en Scopus; NEAR/n en Web of Science. En IEEE Xplore, la cadena principal usa AND (NEAR no soportado), y la cadena puente se expandió en combinaciones explícitas por pares NEAR/n.",
        "Inglés y español",
        "2018–2026",
        "Scopus, IEEE Xplore y Web of Science"
    ]
}

df_search_params = pd.DataFrame(data)
tt.display_styled_df(df=df_search_params)

Aspecto,Especificación
Cadena principal,"(""Brent"" OR ""crude oil"" OR ""oil price"" OR ""energy market"") W/15 (""volatility"" OR ""realized volatility"" OR ""variance forecasting"" OR ""GARCH"") W/15 (""deep learning"" OR ""neural network"" OR ""LSTM"" OR ""Transformer"" OR ""CNN"" OR ""hybrid model"") W/15 (""forecasting"" OR ""prediction"" OR ""out-of-sample"")"
Cadena puente,"(""Brent"" OR ""crude oil"" OR ""oil price"" OR ""energy market"") W/15 (""deep learning"" OR ""neural network"" OR ""LSTM"" OR ""Transformer"" OR ""CNN"" OR ""hybrid model"") W/15 (""forecasting"" OR ""prediction"" OR ""out-of-sample"") AND (""deep learning"" OR ""neural network"") W/10 (""financial time series"" OR ""volatility"")"
Operadores de proximidad,"W/n en Scopus; NEAR/n en Web of Science. En IEEE Xplore, la cadena principal usa AND (NEAR no soportado), y la cadena puente se expandió en combinaciones explícitas por pares NEAR/n."
Idiomas,Inglés y español
Rango temporal,2018–2026
Bases de datos,"Scopus, IEEE Xplore y Web of Science"


### Criterios de inclusión y exclusión

Para que un registro fuera elegible en el corpus, debía cumplir simultáneamente tres condiciones generales de inclusión (C1–C3). La condición C1 exige relevancia directa al problema de pronóstico de volatilidad del crudo o activos financieros transferibles, empleando arquitecturas de aprendizaje profundo como componente central. La condición C2 evalúa la reproducibilidad intrínseca del estudio, requiriendo métricas claras y suficiente detalle arquitectónico o de hiperparámetros para reconstruir el protocolo. Finalmente, la condición C3 filtra por calidad editorial, admitiendo publicaciones en revistas Q1 o Q2 (WoS o Scopus) o conferencias de prestigio, con excepciones justificadas únicamente para trabajos fundacionales en revistas de menor cuartil.

Por otro lado, un registro fue excluido del corpus si presentaba alguna de las condiciones (E1–E7). Estas condiciones capturan limitaciones de acceso, deficiencias metodológicas severas o desalineación con el objetivo central del benchmark.

In [7]:
data_exclusion = {
    "Código": ["E1", "E2", "E3", "E4", "E5", "E6", "E7"],
    "Criterio de exclusión": [
        "Duplicado de otro registro ya identificado.",
        "Tipo de documento no evaluable (editorial, reseña, solo resumen).",
        "Idioma distinto de inglés o español.",
        "Texto completo no accesible tras intentos razonables.",
        "Protocolo de evaluación no reconstruible.",
        "Aplicación de deep learning únicamente al nivel de precio sin pronóstico de volatilidad, varianza realizada ni proxy equivalente.",
        "Modelos econométricos clásicos de la familia GARCH como contribución principal (permitidos solo como modelo de referencia)."
    ]
}

df_exclusion = pd.DataFrame(data_exclusion)
tt.display_styled_df(df=df_exclusion)

Código,Criterio de exclusión
E1,Duplicado de otro registro ya identificado.
E2,"Tipo de documento no evaluable (editorial, reseña, solo resumen)."
E3,Idioma distinto de inglés o español.
E4,Texto completo no accesible tras intentos razonables.
E5,Protocolo de evaluación no reconstruible.
E6,"Aplicación de deep learning únicamente al nivel de precio sin pronóstico de volatilidad, varianza realizada ni proxy equivalente."
E7,Modelos econométricos clásicos de la familia GARCH como contribución principal (permitidos solo como modelo de referencia).


### Evaluación de riesgo de sesgo

La calidad metodológica de los estudios incluidos se evaluó mediante un instrumento diseñado *ad hoc* compuesto por seis dominios generales de sesgo (D1–D6). Este instrumento no pretende ser un estándar clínico validado, sino una herramienta trazable para comparar la rigurosidad del corpus. Cada dominio se evalúa mediante una pregunta de juicio que se responde con una escala cualitativa. 

Para cuantificar la evaluación en la matriz de extracción, se asigna un valor numérico a cada respuesta: No (0) para sesgo ausente o controlado, Tal vez (1) para información no reportada o no verificable, y Sí (2) cuando hay evidencia de sesgo o el estudio no permite descartarlo. Es importante destacar que esta escala se utiliza de forma puramente descriptiva por dominio; no se calcula un puntaje sumativo total, sino que se reportan los valores individuales para cada criterio. 

Para garantizar la trazabilidad, cada respuesta en la matriz de extracción se acompaña de una justificación breve que cita la sección, figura o pseudocódigo del estudio original, de modo que la evaluación no dependa de una impresión subjetiva.

In [11]:
data_sesgo = {
    "Código": ["D1", "D2", "D3", "D4", "D5", "D6"],
    "Dominio de sesgo": [
        "Selección y partición", 
        "Fuga de información", 
        "Reporte selectivo", 
        "Medición e inferencia", 
        "Comparación entre modelos", 
        "Reproducibilidad"
    ],
    "Pregunta de juicio": [
        "¿La partición de los datos permite descartar que el desempeño dependa de una división favorable o no reproducible? (Cronológica vs. aleatoria).",
        "¿El preprocesamiento, la selección de atributos y el ajuste de hiperparámetros se realizaron solo con información anterior al origen de pronóstico?",
        "¿Se reportan todos los resultados relevantes (varianza, fallos) o solo los favorables?",
        "¿La evaluación incluye pruebas de significancia formal entre modelos, o se basa solo en métricas puntuales?",
        "¿Los baselines están ajustados bajo condiciones equivalentes (presupuesto de cómputo, partición)?",
        "¿El estudio reporta arquitectura, hiperparámetros clave y protocolo de ajuste suficientes para reimplementar el modelo?"
    ]
}


df_sesgo = pd.DataFrame(data_sesgo)
tt.display_styled_df(df=df_sesgo)

Código,Dominio de sesgo,Pregunta de juicio
D1,Selección y partición,¿La partición de los datos permite descartar que el desempeño dependa de una división favorable o no reproducible? (Cronológica vs. aleatoria).
D2,Fuga de información,"¿El preprocesamiento, la selección de atributos y el ajuste de hiperparámetros se realizaron solo con información anterior al origen de pronóstico?"
D3,Reporte selectivo,"¿Se reportan todos los resultados relevantes (varianza, fallos) o solo los favorables?"
D4,Medición e inferencia,"¿La evaluación incluye pruebas de significancia formal entre modelos, o se basa solo en métricas puntuales?"
D5,Comparación entre modelos,"¿Los baselines están ajustados bajo condiciones equivalentes (presupuesto de cómputo, partición)?"
D6,Reproducibilidad,"¿El estudio reporta arquitectura, hiperparámetros clave y protocolo de ajuste suficientes para reimplementar el modelo?"


### Reproducibilidad operativa por el equipo

La reproducibilidad metodológica del paper (evaluada en C2) y la reproducibilidad operativa para el benchmark del proyecto son dimensiones distintas. Un estudio puede ser metodológicamente impecable y, sin embargo, no ser reproducible con los recursos del equipo (por ejemplo, por requerir variables propietarias o un clúster de GPU inaccesible). Dichos estudios no pierden mérito en la revisión ni se excluyen del corpus; simplemente no entran al conjunto benchmark.

La reproducibilidad operativa no exige acceder a los datos idénticos del paper, sino a las variables necesarias (entradas y target) mediante fuentes públicas equivalentes. Dado que el benchmark se ejecuta sobre el dataset del proyecto (Brent), lo que debe coincidir es el conjunto de variables y el protocolo, no las filas exactas. Para evaluar esta dimensión, se establecieron tres criterios operativos fundamentales.

Respecto al acceso de datos, un proxy público se considera aceptable si mide el mismo constructo, cubre el mismo período y se declara explícitamente en la ficha del modelo. A partir de esta evaluación, un estudio se clasifica como Reproducible por el equipo (R-A) si existe acceso a las variables requeridas, si es posible reconstruir el pipeline completo y si su viabilidad computacional es positiva o parcial. En caso contrario, se clasifica como No reproducible por el equipo (R-B) y se conserva en la revisión como referencia arquitectónica, sin competir en el benchmark.

In [9]:
data_reproducibilidad = {
    "Código": ["R1", "R2", "R3"],
    "Ítem": [
        "Acceso a variables", 
        "Reconstructibilidad", 
        "Viabilidad computacional"
    ],
    "Pregunta operativa": [
        "¿Se puede acceder a las variables de entrada y target (originales o proxies públicos equivalentes) para reconstruir el modelo?",
        "¿Se puede reconstruir el pipeline completo (preprocesamiento, arquitectura, ajuste) con los detalles reportados?",
        "¿El modelo puede entrenarse y evaluarse dentro del presupuesto de cómputo del proyecto?"
    ]
}

df_reproducibilidad = pd.DataFrame(data_reproducibilidad)
tt.display_styled_df(df=df_reproducibilidad)

Código,Ítem,Pregunta operativa
R1,Acceso a variables,¿Se puede acceder a las variables de entrada y target (originales o proxies públicos equivalentes) para reconstruir el modelo?
R2,Reconstructibilidad,"¿Se puede reconstruir el pipeline completo (preprocesamiento, arquitectura, ajuste) con los detalles reportados?"
R3,Viabilidad computacional,¿El modelo puede entrenarse y evaluarse dentro del presupuesto de cómputo del proyecto?


### Proceso de cribado

El proceso para construir el corpus final siguió el protocolo PRISMA 2020 y se ejecutó en cuatro etapas secuenciales:

1. **Filtros iniciales y Rayyan:** Las búsquedas en Scopus, IEEE Xplore y Web of Science se restringieron a artículos de revista, ponencias, idiomas (inglés/español) y el rango temporal 2018–2026. Los resultados se exportaron a Rayyan para el cribado.
2. **Cribado en dos fases:** En la primera, se evaluaron títulos y resúmenes aplicando los criterios de inclusión (C1–C3) y exclusión (E1–E7). En la segunda, se evaluaron los textos completos registrando el motivo específico de exclusión.
3. **Control de calidad:** Un segundo revisor verificó de forma independiente una muestra aleatoria del 10 % de las decisiones de título/resumen y el 100 % de las exclusiones de texto completo. Las discrepancias se resolvieron por consenso, priorizando la inclusión en casos ambiguos.
4. **Extracción de datos:** Se construyó una matriz estructurada con las variables clave de cada estudio (autores, familia arquitectónica, target, partición, métricas, código, etc.), verificada en su totalidad por el supervisor.

La búsqueda definitiva se ejecutó el 31 de octubre de 2026. Sobre los registros incluidos se aplicaron dos evaluaciones independientes. Primero, una evaluación de riesgo de sesgo (D1–D6) con justificación por dominio, donde el puntaje total se usa únicamente para clasificar el riesgo (bajo, moderado, alto) durante el cribado; en la tabla comparativa final del Top 10 solo se reportan los valores descriptivos (0, 1 o 2) sin sumatorias. Segundo, una evaluación de reproducibilidad operativa (R1–R3) que clasifica cada estudio como reproducible por el equipo (R-A) o no reproducible (R-B). 

Cabe destacar que no se aplicaron instrumentos formales de certeza (tipo GRADE) porque esta síntesis es de naturaleza narrativa y no agrega estimaciones cuantitativas. La evaluación de sesgo es una herramienta *ad hoc* diseñada específicamente para este ejercicio, y se declara como tal.

### Regla de decisión para el conjunto benchmark

La inclusión de un modelo en el benchmark no depende solo de su calidad metodológica, sino del cruce entre su riesgo de sesgo y su reproducibilidad operativa. La lógica es simple: un estudio impecable con datos propietarios no puede competir en igualdad de condiciones, mientras que un estudio con fuga de información sí puede, siempre que el equipo sea capaz de reconstruirlo y corregirlo. Este cruce de dimensiones se resume en la siguiente matriz de decisión.

In [10]:
data_decision = {
    "Nivel de Riesgo": [
        "Riesgo bajo / moderado", 
        "Riesgo alto"
    ],
    "Reproducible (R-A)": [
        "Candidato directo", 
        "Candidato con corrección"
    ],
    "No reproducible (R-B)": [
        "Referencia arquitectónica", 
        "Excluido del benchmark"
    ]
}

df_decision = pd.DataFrame(data_decision)
tt.display_styled_df(df=df_decision)

Nivel de Riesgo,Reproducible (R-A),No reproducible (R-B)
Riesgo bajo / moderado,Candidato directo,Referencia arquitectónica
Riesgo alto,Candidato con corrección,Excluido del benchmark


Para garantizar la equidad experimental y evitar que el benchmark herede los sesgos de la literatura original, se aplican cuatro reglas estrictas:

- **Regla 1:** Ningún estudio con sesgo simultáneo en D1 (partición) y D2 (fuga) entra al benchmark sin aplicar una corrección documentada.
- **Regla 2:** La corrección de la fuga es idéntica para todos los modelos (mismo protocolo de validación cronológica).
- **Regla 3:** El desempeño original del paper no se usa como referencia para el ranking; solo se contrasta contra el "desempeño corregido" para medir el impacto de la fuga como un hallazgo empírico.
- **Regla 4:** Los estudios clasificados como No reproducibles (R-B) no se descartan del corpus; aportan evidencia de la brecha y enriquecen el mapeo de familias arquitectónicas.
  
## Resultados 

La aplicación del protocolo PRISMA 2020 permitió identificar un total de 365 registros iniciales distribuidos entre Scopus (81), IEEE Xplore (246) y Web of Science (38). Tras la eliminación de 71 duplicados y 59 registros descartados automáticamente, se cribaron 235 títulos y resúmenes. De estos, 35 fueron buscados para recuperación de texto completo (4 no recuperables) y 31 fueron evaluados en profundidad. Finalmente, el corpus se consolidó con 16 estudios incluidos. Los resúmenes estructurados y metadatos de estos 35 estudios recuperados se encuentran disponibles en formato Markdown (`.md`) dentro de la carpeta `literature/` del repositorio del proyecto, garantizando la trazabilidad de la extracción.

<div align="center">
  <img src="../diagrams/prisma_flow_chart.svg" alt="Diagrama PRISMA" width="60%">
</div>

La exclusión de los 15 estudios restantes respondió a tres causas principales. La más frecuente fue el desajuste del objetivo de pronóstico (n=11), donde los modelos predecían el nivel de precio o retornos en lugar de la volatilidad o varianza. La segunda causa fue la baja calidad editorial o la falta de carácter fundacional (n=3), y finalmente, un estudio fue excluido por presentar un protocolo de evaluación no reconstruible al carecer de partición de prueba (n=1).

### Resultados generales

La matriz de extracción consolida las decisiones técnicas y de evaluación de los 16 estudios incluidos. La siguiente tabla presenta un resumen de la arquitectura, la variable objetivo, el esquema de partición y el proceso de ajuste. El repositorio del proyecto aloja la matriz completa, la cual incluye las funciones de pérdida, las métricas de error y la evaluación de sesgo por estudio.

In [64]:
data_extraccion = {
    "Estudio": [
        "Yu2026", "Liu2026", "Xing2026", "Romdhane2026", 
        "Gong2025", "Pan2025", 
        "Zhang2024a", "Zhang2024b", "Xu2024", "Gadhi2024", "Ouyang2024", 
        "Jiang2023", 
        "Kakade2022", "Jiao2022", 
        "Li2021", 
        "AlFattah2019"
    ],
    "Arquitectura": [
        "Desc.–TCN", "LSTM–TSMixer", "SVMD-CBAM-BiGRU-TCN", "HAR-LSTM-GARCH", 
        "STN-convLSTM", "WTD-HAR-GRU", 
        "Transformer–CNN", "Desc.–Reservorio", "Bi-LSTM-Attention", "GARCH–WGAN-GP", "BiLSTM-Attention", 
        "VMD-ICEEMDAN-LSTM", 
        "GARCH–LSTM Ensemble", "PSO-LSTM + NLP", 
        "VMD-BiLSTM + NLP", 
        "MLP + GA"
    ],
    "Variable Objetivo": [
        "WTI RV mensual", "INE RV diaria", "WTI y Brent RV", "Brent RV proxy", 
        "WTI y Henry Hub RV", "INE RV diaria", 
        "WTI RV mensual", "Retornos cuadráticos", "Retornos absolutos", "Brent Std (5-días)", "WTI Varianza Cond.", 
        "INE RV diaria", 
        "WTI VaR", "WTI Std (5-días)", 
        "Brent Std (7-días)", 
        "Retornos cuadráticos"
    ],
    "Partición": [
        "Estática", "Estática", "Cronológica 72/8/20", "Cronológica 70/15/15", 
        "Cronológica 80/20", "Estática 70/30", 
        "Rolling origin", "Estática", "Estática", "No declarada", "Estática 80/20", 
        "Cronológica 7:2:1", 
        "Estática", "Cronológica", 
        "Cronológica 90/10", 
        "Cronológica 70/15/15"
    ],
    "Proceso de Tuning": [
        "No reportado", "Parcial", "Sí", "Sí", 
        "No reportado", "No", 
        "Parcial", "Sí (WOA)", "No reportado", "No reportado", "No reportado", 
        "No", 
        "No reportado", "Sí (PSO)", 
        "No", 
        "Parcial"
    ]
}

df_extraccion = pd.DataFrame(data_extraccion)

tt.display_styled_df(df=df_extraccion)

Estudio,Arquitectura,Variable Objetivo,Partición,Proceso de Tuning
Yu2026,Desc.–TCN,WTI RV mensual,Estática,No reportado
Liu2026,LSTM–TSMixer,INE RV diaria,Estática,Parcial
Xing2026,SVMD-CBAM-BiGRU-TCN,WTI y Brent RV,Cronológica 72/8/20,Sí
Romdhane2026,HAR-LSTM-GARCH,Brent RV proxy,Cronológica 70/15/15,Sí
Gong2025,STN-convLSTM,WTI y Henry Hub RV,Cronológica 80/20,No reportado
Pan2025,WTD-HAR-GRU,INE RV diaria,Estática 70/30,No
Zhang2024a,Transformer–CNN,WTI RV mensual,Rolling origin,Parcial
Zhang2024b,Desc.–Reservorio,Retornos cuadráticos,Estática,Sí (WOA)
Xu2024,Bi-LSTM-Attention,Retornos absolutos,Estática,No reportado
Gadhi2024,GARCH–WGAN-GP,Brent Std (5-días),No declarada,No reportado


La matriz expone dos características metodológicas. Primero, los esquemas de partición de datos difieren entre estudios. `Yu2026` y `Liu2026` aplican particiones estáticas, las cuales evalúan el modelo bajo condiciones fijas sin contemplar cambios de régimen posteriores al entrenamiento. `Jiang2023` y `Romdhane2026` utilizan particiones cronológicas simples, mientras que `Zhang2024a` emplea validación de origen móvil (rolling origin). Esta disparidad afecta directamente las métricas reportadas. Las particiones estáticas no emulan un entorno de pronóstico en tiempo real. La ausencia de un esquema de evaluación temporal común impide atribuir las diferencias de rendimiento a la capacidad predictiva de la arquitectura del modelo.

Segundo, el reporte del ajuste de hiperparámetros varía entre los estudios. `Yu2026`, `Gong2025` y `Xu2024` omiten el proceso de selección. `Liu2026` y `AlFattah2019` reportan el proceso parcialmente. `Zhang2024b` y `Jiao2022 `aplican optimizadores bioinspirados (WOA y PSO). `Xing2026` y `Romdhane2026` declaran búsquedas sistemáticas. Un presupuesto de ajuste desigual distorsiona las evaluaciones comparativas. Los modelos que reciben una optimización exhaustiva superan a aquellos que utilizan parámetros por defecto, independientemente de la ventaja estructural de la arquitectura. La falta de un protocolo de ajuste estandarizado invalida la comparación directa de rendimiento entre los estudios del corpus.

El West Texas Intermediate (WTI) concentra la mitad de los estudios del corpus, mientras que una minoría analiza exclusivamente el crudo Brent. Adicionalmente, la categoría Múltiples agrupa los modelos que procesan series de varios activos simultáneamente, específicamente combinaciones de WTI con Brent o Henry Hub.

In [55]:
activos = [
    'WTI', 
    'INE (China)', 
    'Brent', 
    'Múltiples'
]
cantidades = [8, 4, 2, 2]
total_estudios = sum(cantidades)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.bar(activos, cantidades, color='white', edgecolor='black', linewidth=1.2, width=0.6)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color('black')
ax.grid(False)

ax.tick_params(axis='both', which='major', labelsize=11, direction='out', length=4, width=0.8)
ax.set_ylabel('Cantidad de estudios', fontsize=12, labelpad=10)
ax.set_title('Distribución de estudios por activo financiero', fontsize=12, pad=15)

ax.set_ylim(0, 10) 
ax.yaxis.set_major_locator(MultipleLocator(1))

for i, v in enumerate(cantidades):
    porcentaje = (v / total_estudios) * 100
    ax.text(i, v + 0.2, f"{porcentaje:.1f}%", ha='center', va='bottom', fontsize=11)

plt.tight_layout()

ts.render_figure(fig, fmt="svg")

La preferencia por el WTI en la literatura responde a la disponibilidad de datos intradía de alta liquidez en la bolsa de Nueva York (NYMEX), lo que facilita el cálculo de la volatilidad realizada. Por su parte, el Brent se comercia como un crudo transportado por vía marítima en la bolsa ICE, por lo que su dinámica de precios responde a la logística global y a eventos geopolíticos internacionales. Mientras que el WTI, al ser un crudo terrestre almacenado en Cushing, Oklahoma, refleja cuellos de botella logísticos locales y políticas domésticas de Estados Unidos. Estas diferencias estructurales en la formación de precios impiden asumir la transferibilidad directa de los modelos predictivos de un mercado a otro y exigen la construcción de un benchmark enfocado en el crudo Brent.

La medición de la volatilidad en el corpus presenta una limitación vinculada a esta disponibilidad de datos. La volatilidad realizada intradía se concentra en los estudios sobre WTI, aprovechando la liquidez de sus precios de alta frecuencia. Para el resto del corpus, los autores emplean estimadores sesgados de la volatilidad, como la desviación estándar móvil de los retornos o la magnitud de los mismos. Adicionalmente, algunos trabajos utilizan el retorno al cuadrado. Aunque este último es un estimador insesgado de la varianza, exhibe una alta varianza estadística (ruido), lo que lo convierte en una medida imprecisa e ineficiente de la volatilidad condicional.

Un problema distinto ocurre con la categoría de varianza condicional, la cual presenta una circularidad metodológica. El estudio `Ouyang2024` calcula la varianza mediante un modelo GARCH y entrena una red neuronal para predecir este valor. La red neuronal aprende las ecuaciones del GARCH en lugar de modelar los patrones de volatilidad del mercado. Si el cálculo del GARCH utiliza datos del conjunto de prueba, el modelo incorpora una fuga de información directa.

In [59]:
proxies = [
    'RV (Intradía)', 
    'Std Móvil', 
    'Retornos²', 
    'Retornos Abs.', 
    'Var. Cond', 
    'EWMA / VaR'
]

cantidades = [7, 4, 2, 1, 1, 1]
total_estudios = sum(cantidades)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.bar(proxies, cantidades, color='white', edgecolor='black', linewidth=1.2, width=0.6)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color('black')
ax.grid(False)

ax.tick_params(axis='both', which='major', labelsize=11, direction='out', length=4, width=0.8)
ax.set_ylabel('Cantidad de estudios', fontsize=12, labelpad=15)
ax.set_title('Distribución de estudios por proxy de volatilidad', fontsize=12, pad=15)

ax.set_ylim(0, 9) 

for i, v in enumerate(cantidades):
    porcentaje = (v / total_estudios) * 100
    ax.text(i, v + 0.2, f"{porcentaje:.1f}%", ha='center', va='bottom', fontsize=11)

plt.tight_layout()

ts.render_figure(fig, fmt="svg")

Los estudios restantes emplean medidas de riesgo en lugar de estimadores de volatilidad. `Kakade2022` utiliza un promedio móvil exponencial (EWMA). El EWMA pondera los retornos históricos de forma decreciente, lo que suaviza la serie y retrasa la detección de cambios abruptos en la volatilidad. Este estudio también utiliza el Valor en Riesgo (VaR) como variable objetivo. El VaR representa un cuantil de la distribución de pérdidas, no una medida de dispersión. Predecir el VaR introduce un error de especificación del objetivo, ya que confunde una medida de riesgo de cola con la volatilidad condicional del activo.

Los estudios de Deep Learning incluidos en esta revisión omiten estimadores insesgados basados en rangos de precios (OHLC), como los propuestos por Parkinson, Garman-Klass y Rogers-Satchell. Estos estimadores presentan una mayor eficiencia estadística que los retornos al cuadrado. Además, requieren únicamente información diaria limitada (apertura, máximo, mínimo y cierre), a diferencia de los estimadores intradía que exigen datos de alta frecuencia. La ausencia de estos estimadores de rango en el corpus establece una brecha metodológica en la construcción de la variable objetivo.

El análisis de las familias arquitectónicas evidenció la preferencia por los diseños híbridos. La familia recurrente (RNN) y con atención dominó el corpus, seguida por los híbridos de econometría y aprendizaje profundo, y los modelos convolucionales (CNN) y temporales. Los enfoques de descomposición y los Transformers presentaron una representación menor, mientras que el perceptrón multicapa (MLP) registró la menor frecuencia.

In [61]:
familias = [
    'RNN / Attn', 
    'Eco + DL', 
    'CNN / TCN', 
    'Desc. / Ens.', 
    'Transformers', 
    'MLP'
]

cantidades = [6, 3, 3, 2, 1, 1]
total_estudios = sum(cantidades)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.bar(familias, cantidades, color='white', edgecolor='black', linewidth=1.2, width=0.6)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color('black')
ax.grid(False)

ax.tick_params(axis='both', which='major', labelsize=11, direction='out', length=4, width=0.8)
ax.set_ylabel('Cantidad de estudios', fontsize=12, labelpad=15)
ax.set_title('Distribución de modelos por familia arquitectónica', fontsize=12, pad=15)

ax.set_ylim(0, 8) 

for i, v in enumerate(cantidades):
    porcentaje = (v / total_estudios) * 100
    ax.text(i, v + 0.2, f"{porcentaje:.1f}%", ha='center', va='bottom', fontsize=11)

plt.tight_layout()

ts.render_figure(fig, fmt="svg")

Esta distribución reflejó los compromisos de ajuste y rendimiento de cada arquitectura. Las redes recurrentes capturaron dependencias temporales, pero presentaron cuellos de botella computacionales por su procesamiento secuencial. Los modelos convolucionales y temporales procesaron patrones locales en paralelo, aunque requirieron mecanismos adicionales para capturar dependencias de largo plazo. Los Transformers resolvieron el problema de las secuencias largas mediante atención global, pero demandaron grandes volúmenes de datos y recursos computacionales, lo que incrementó el riesgo de sobreajuste en series financieras limitadas.

Ningún estudio del corpus empleó una arquitectura pura como contribución principal. La totalidad de los trabajos integró al menos dos componentes, como descomposiciones de señales o variables exógenas, para elevar la capacidad de ajuste del modelo. Esta estrategia de hibridación incrementó el número de parámetros y la complejidad estructural. La ausencia de modelos base simples impidió aislar el rendimiento que aportó la arquitectura neuronal frente a la ganancia que generó el preprocesamiento adicional.

La evaluación del rendimiento en el corpus dependió de métricas de error de regresión tradicionales. Los estudios omitieron el uso de métricas ajustadas por heterocedasticidad y la pérdida de cuasi-verosimilitud (Quasi-Likelihood, QLIKE), las cuales operan como el estándar para penalizar errores en periodos de alta volatilidad. Esta elección limitó la medición de la utilidad de los modelos para la gestión de riesgo financiero.

In [62]:
metricas = [
    'Error Estándar\n(MAE/MSE)', 
    'Ajustadas\n(HMSE/HMAE)', 
    'Intervalos\n(PICP/PINAW)', 
    'QLIKE'
]

cantidades = [16, 4, 2, 1]
total_estudios = 16

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.bar(metricas, cantidades, color='white', edgecolor='black', linewidth=1.2, width=0.6)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color('black')
ax.grid(False)

ax.tick_params(axis='both', which='major', labelsize=11, direction='out', length=4, width=0.8)
ax.set_ylabel('Cantidad de estudios', fontsize=12, labelpad=15)
ax.set_title('Métricas de evaluación', fontsize=12, pad=15)

ax.set_ylim(0, 18) 

for i, v in enumerate(cantidades):
    porcentaje = (v / total_estudios) * 100
    ax.text(i, v + 0.3, f"{porcentaje:.1f}%", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
ts.render_figure(fig, fmt="svg")

La validación estadística entre arquitecturas presentó deficiencias. Solo 10 de los 16 estudios aplicaron pruebas formales, como el Model Confidence Set o el test de Diebold-Mariano, para verificar si las diferencias de rendimiento fueron significativas. Los seis estudios restantes omitieron la comparación directa o se limitaron a pruebas de cobertura de VaR. Adicionalmente, los estudios que aplicaron pruebas formales utilizaron umbrales de significancia distintos (5%, 20% y 25%). La ausencia de un nivel de significancia común impidió la comparación directa de los p-valores reportados. La falta de un protocolo de evaluación unificado justificó la construcción del benchmark para someter a todas las arquitecturas al mismo rigor estadístico.

La heterogeneidad metodológica del corpus impide la comparación directa de los rendimientos reportados. Las diferencias en la construcción del target, el ajuste de hiperparámetros, la complejidad arquitectónica y la validación estadística distorsionan el ranking de los modelos. Estas limitaciones exigen una evaluación sistemática de la validez de cada estudio. La siguiente sección presenta el análisis de riesgo de sesgo de los 16 trabajos incluidos. El análisis evalúa seis dominios específicos para identificar las fuentes de fuga de información y las comparaciones desiguales que afectan la literatura.

### Análisis de riesgo de sesgo

La evaluación de la calidad metodológica de los 16 estudios incluidos se sintetiza en una matriz de seis dominios de sesgo. La tabla presenta el resumen de los resultados por cada trabajo.

El análisis de la matriz identifica la fuga de información (D2) como el sesgo más frecuente. Nueve estudios aplicaron técnicas de descomposición de señales (VMD, EMD o Wavelets) o procesos de normalización sobre el conjunto de datos completo antes de dividir en entrenamiento y prueba. Esta secuencia de pasos transfirió información del futuro hacia el pasado. Los modelos incorporaron las propiedades matemáticas del conjunto de prueba en la fase de entrenamiento. Este flujo de datos produjo errores cuadráticos medios (MSE) sin validez predictiva, como el valor de $3.81 \times 10^{-9}$ registrado por `Gong2025` y el de $3.60 \times 10^{-10}$ registrado por `Pan2025`.

In [63]:
data_sesgos = {
    "Estudio": [
        "Yu2026", "Liu2026", "Xing2026", "Romdhane2026", "Gong2025", 
        "Pan2025", "Zhang2024a", "Zhang2024b", "Xu2024", "Gadhi2024", 
        "Ouyang2024", "Jiang2023", "Kakade2022", "Jiao2022", "Li2021", 
        "AlFattah2019"
    ],
    "D1": [1, 1, 0, 0, 1, 0, 0, 1, 1, 2, 0, 1, 1, 0, 0, 0],
    "D2": [2, 1, 0, 0, 2, 2, 0, 2, 1, 2, 2, 2, 2, 1, 2, 1],
    "D3": [1, 1, 0, 0, 1, 0, 1, 2, 1, 2, 0, 2, 1, 1, 0, 2],
    "D4": [0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 1, 2, 0, 1],
    "D5": [1, 1, 1, 0, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2],
    "D6": [2, 1, 0, 1, 1, 2, 2, 2, 2, 2, 1, 2, 1, 1, 1, 1]
}

df_sesgos = pd.DataFrame(data_sesgos).set_index("Estudio")
df_sesgos.index.name = None 

def color_bias(val):
    if val == 0:
        color = '#2ecc71'
    elif val == 1:
        color = '#f1c40f'
    elif val == 2:
        color = '#e74c3c'
    else:
        color = 'white'
    return f'background-color: {color};'

styled_df = (
     df_sesgos.style
    .map(color_bias)               
    .format(lambda x: "")
    .set_table_styles(tt._table_style_rules())
)

tt._render_styled(styled_df)

,D1,D2,D3,D4,D5,D6
Yu2026,,,,,,
Liu2026,,,,,,
Xing2026,,,,,,
Romdhane2026,,,,,,
Gong2025,,,,,,
Pan2025,,,,,,
Zhang2024a,,,,,,
Zhang2024b,,,,,,
Xu2024,,,,,,
Gadhi2024,,,,,,


El dominio de partición (D1) y el de reporte selectivo (D3) presentaron deficiencias. Varios estudios omitieron el orden cronológico o no declararon los esquemas de partición estáticos. Otros autores reportaron únicamente los mejores resultados y omitieron la varianza entre ejecuciones o las pérdidas de los modelos base.

El dominio de comparación entre modelos (D5) registró sesgo. Las comparaciones no operaron bajo condiciones equivalentes. Los modelos econométricos procesaron series univariadas, mientras que las redes profundas incorporaron variables exógenas o predicciones de otros submodelos. El presupuesto de cómputo y la intensidad del ajuste de hiperparámetros difirieron. Estas diferencias de información y recursos favorecieron a los modelos profundos.

Los dominios de inferencia estadística (D4) y reproducibilidad (D6) presentaron un control deficiente. Solo 10 estudios aplicaron pruebas de significancia formal. Los umbrales de significancia variaron entre 5%, 20% y 25%, lo que impidió la comparación de los *p*-valores. Más aún, La falta de código público (presente solo en `Xing2026`) y la omisión de hiperparámetros limitan la reproducción de los experimentos. 

El conjunto de estos hallazgos sugieren que las mejoras de rendimiento reportadas en la literatura reflejan diferencias en la información de entrada y fallas en el flujo de datos, en lugar de ventajas estructurales de las arquitecturas neuronales.

### Definición del conjunto benchmark

El benchmark define los retornos al cuadrado del crudo Brent como el único insumo para todos los modelos. Esta restricción excluye las arquitecturas que operan con variables exógenas o Procesamiento de Lenguaje Natural. Al homogeneizar la información de entrada, el diseño aísla la capacidad predictiva de la arquitectura frente a las ventajas que otorga el uso de información adicional.

La conformación del conjunto responde a un estudio de ablación progresivo. El benchmark integra cuatro modelos de referencia (baselines): GARCH, EGARCH, HAR y HAR-WTD. Estos modelos establecen el piso de rendimiento para los métodos estadísticos y de filtrado de señales. El grupo principal corresponde a 10 modelos de Deep Learning, organizados de menor a mayor complejidad. La progresión inicia con los bloques base (LSTM, GRU y TCN) y avanza hacia variantes bidireccionales (BiLSTM, BiGRU), módulos de atención (BiLSTM-Attention) y fusiones arquitectónicas (LSTM-TSMixer, BiGRU-TCN). El conjunto finaliza con híbridos que integran los baselines con redes neuronales (GARCH-LSTM y WTD-HAR-GRU). Esta estructura permite aislar el aporte real de cada componente metodológico.

In [83]:
data = [
    ["GARCH", "Baseline", "Econométrico", "BiLSTM", "Deep Learning", "Recurrente Bidireccional"],
    ["EGARCH", "Baseline", "Econométrico", "BiGRU", "Deep Learning", "Recurrente Bidireccional"],
    ["HAR", "Baseline", "Econométrico", "BiLSTM-Attention", "Deep Learning", "Atención"],
    ["HAR-WTD", "Baseline", "Descomposición", "LSTM-TSMixer", "Deep Learning", "Híbrido Recurrente-MLP"],
    ["LSTM", "Deep Learning", "Recurrente", "BiGRU-TCN", "Deep Learning", "Híbrido CNN-RNN"],
    ["GRU", "Deep Learning", "Recurrente", "GARCH-LSTM", "Deep Learning", "Econométrico + DL"],
    ["TCN", "Deep Learning", "Convolucional", "WTD-HAR-GRU", "Deep Learning", "Wavelets + Econométrico + DL"]
]

columns = ["Modelo", "Grupo", "Familia", "Modelo", "Grupo", "Familia"]

df_benchmark = pd.DataFrame(data, columns=columns)

tt.display_styled_df(df=df_benchmark)

Modelo,Grupo,Familia,Modelo,Grupo,Familia
GARCH,Baseline,Econométrico,BiLSTM,Deep Learning,Recurrente Bidireccional
EGARCH,Baseline,Econométrico,BiGRU,Deep Learning,Recurrente Bidireccional
HAR,Baseline,Econométrico,BiLSTM-Attention,Deep Learning,Atención
HAR-WTD,Baseline,Descomposición,LSTM-TSMixer,Deep Learning,Híbrido Recurrente-MLP
LSTM,Deep Learning,Recurrente,BiGRU-TCN,Deep Learning,Híbrido CNN-RNN
GRU,Deep Learning,Recurrente,GARCH-LSTM,Deep Learning,Econométrico + DL
TCN,Deep Learning,Convolucional,WTD-HAR-GRU,Deep Learning,Wavelets + Econométrico + DL


La literatura del corpus implementa las arquitecturas base de forma individual (LSTM, GRU, TCN). Sin embargo, las mejoras reportadas por los modelos monolíticos complejos carecen de fiabilidad metodológica. Esta falta de fiabilidad responde a que sus diseños no permiten aislar el origen de la mejora predictiva. Por ejemplo, el modelo de `Jiang2023` confunde el efecto de la descomposición secundaria con el uso de hasta 17 redes LSTM en paralelo. De igual forma, arquitecturas como el SVMD-CBAM-BiGRU-TCN integran descomposiciones y módulos de atención que actúan como cajas negras, lo que imposibilita una auditoría limpia de fugas de información. 

Para establecer un control metodológico superior, el benchmark descompone estas arquitecturas masivas en sus bloques fundamentales (TCN, BiGRU, LSTM). Esta decisión garantiza la ausencia de efectos confusores y permite un estudio de ablación limpio. Los modelos de descomposición que sí ingresaron (`Xing2026` y `Pan2025`) se reimplementan aplicando sus respectivas transformadas mediante ventanas móviles, garantizando que el entrenamiento procese únicamente datos históricos disponibles en el momento del pronóstico.